In [1]:
!pip install linkedin-jobs-scraper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 31.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import files
import shutil
import pandas as pd
import logging
from datetime import datetime
from linkedin_jobs_scraper import LinkedinScraper
from linkedin_jobs_scraper.events import Events, EventData, EventMetrics
from linkedin_jobs_scraper.query import Query, QueryOptions, QueryFilters
from linkedin_jobs_scraper.filters import RelevanceFilters, TimeFilters, TypeFilters, ExperienceLevelFilters
import openai
import time
import traceback
from pathlib import Path
import traceback
from typing import Dict, List, Optional, Union
import glob
from openai import OpenAI
from google.colab import userdata
import json




In [3]:

input_dir = 'content/input'
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
else:
    print(f"Directory {input_dir} already exists")

print("Please select multiple files to upload...")
uploaded = files.upload()

for filename in uploaded.keys():
    source_path = filename
    destination_path = os.path.join(input_dir, filename)

    shutil.move(source_path, destination_path)
    print(f"Moved {filename} to {destination_path}")

print("\nFiles in the input directory:")
for file in os.listdir(input_dir):
    file_path = os.path.join(input_dir, file)
    file_size = os.path.getsize(file_path) / 1024  # Size in KB
    print(f"- {file} ({file_size:.2f} KB)")

Created directory: content/input
Please select multiple files to upload...


Saving CV_Aryan_Ishaan.pdf to CV_Aryan_Ishaan.pdf
Saving CV_Ishaan Aryan.docx to CV_Ishaan Aryan.docx
Moved CV_Aryan_Ishaan.pdf to content/input/CV_Aryan_Ishaan.pdf
Moved CV_Ishaan Aryan.docx to content/input/CV_Ishaan Aryan.docx

Files in the input directory:
- CV_Aryan_Ishaan.pdf (151.35 KB)
- CV_Ishaan Aryan.docx (20.12 KB)


In [4]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('linkedin_scraper')


def scrape_linkedin_jobs(search_query: str,location_list: List[str], job_limit: int = 20,experience_levels: List[ExperienceLevelFilters] = None,job_types: List[TypeFilters] = None,
                         time_filter: List[TimeFilters] = None
) -> Dict[str, Union[str, int, List[str]]]:
    """
    Scrapes job postings from LinkedIn based on the provided search query and location list.
    """
    if job_types is None:
        job_types = [TypeFilters.FULL_TIME, TypeFilters.INTERNSHIP]

    if experience_levels is None:
        experience_levels = [
            ExperienceLevelFilters.ENTRY_LEVEL,
            ExperienceLevelFilters.ASSOCIATE
        ]

    if time_filter is None:
        time_filter = [TimeFilters.DAY,TimeFilters.MONTH]

    scraper = None
    try:
        job_data_df = pd.DataFrame(columns=[
            'title', 'company', 'company_link', 'date', 'date_text',
            'link', 'description', 'location'
        ])

        processed_jobs = set()
        jobs_count = 0

        def on_data(data: EventData):
            nonlocal job_data_df, jobs_count, processed_jobs

            if jobs_count >= job_limit:
                return

            if data.link in processed_jobs:
                return

            processed_jobs.add(data.link)

            # Create new row for DataFrame
            new_row = {
                'title': data.title,
                'company': data.company,
                'company_link': data.company_link,
                'date': data.date,
                'date_text': data.date_text,
                'link': data.link,
                'description': data.description,
                'location': data.location
            }

            job_data_df = pd.concat([job_data_df, pd.DataFrame([new_row])], ignore_index=True)
            jobs_count += 1

            logger.info(f"Scraped job {jobs_count}/{job_limit}: {data.title} at {data.company}")

            if jobs_count >= job_limit:
                logger.info(f"Reached job limit of {job_limit}. Stopping scraper.")
                return

        def on_error(error):
            logger.error(f"Scraping error: {error}")

        def on_end():
            logger.info("Scraping completed")

        scraper = LinkedinScraper(
            headless=True,
            max_workers=1,
            slow_mo=0.3,
            page_load_timeout=60
        )

        scraper.on(Events.DATA, on_data)
        scraper.on(Events.ERROR, on_error)
        scraper.on(Events.END, on_end)

        queries = [
            Query(
                query=search_query,
                options=QueryOptions(
                    locations=location_list,
                    limit=job_limit,
                    filters=QueryFilters(
                        relevance=RelevanceFilters.RECENT,
                        time=time_filter,
                        type=job_types,
                        experience=experience_levels
                    )
              )
            )
        ]

        logger.info(f"Starting LinkedIn scraper for '{search_query}' in {location_list}")
        scraper.run(queries)

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"linkedin_jobs_{search_query.replace(' ', '_')}_{timestamp}.csv"

        job_data_df.to_csv(filename, index=False)
        logger.info(f"Saved {len(job_data_df)} jobs to {filename}")

        return {
            "filename": filename,
            "status": "success",
            "job_count": len(job_data_df),
            "search_query": search_query,
            "locations": location_list
        }

    except Exception as e:
        logger.exception(f"Scraping failed: {e}")
        return {
            "status": "error",
            "message": str(e),
            "search_query": search_query,
            "locations": location_list
        }



In [5]:
def score_jobs_against_cv(job_file_name: str, assistant_id: str) -> dict:
    try:
        job_data_df = pd.read_csv(job_file_name)
        print(f"Processing {len(job_data_df)} jobs from {job_file_name}")

        scores = []
        reasons = []
        cover_letters = []
        cv_edits = []

        for idx, row in job_data_df.iterrows():
            job_desc = row['description']
            print(f"Processing job {idx+1}/{len(job_data_df)}")

            thread = client.beta.threads.create()
            try:
                score_prompt = f"""
                Role: You are an experienced recruiter at a top-tier executive placement firm specializing in precision candidate-job matching.
                Task: Analyze the user's CV pdf (use file_search) and the following job posting. Score their fit.

                === Job Description ===
                {job_desc}

                Evaluate relevance, growth potential, and competitive edge. Give the reason in a short 1 line after the score.
                Format your output exactly like this sample:

                Employability Score: 8.5
                Reason: Strong fit in X, Y, and Z areas.
                """
                client.beta.threads.messages.create(
                    thread_id=thread.id, role="user", content=score_prompt
                )
                run = client.beta.threads.runs.create(
                    thread_id=thread.id,
                    assistant_id=assistant_id
                )

                start_time = time.time()
                max_wait_time = 120  # 2 minutes timeout
                while True:
                    if time.time() - start_time > max_wait_time:
                        raise Exception("Run timed out after 2 minutes")

                    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
                    if run_status.status == "completed":
                        break
                    elif run_status.status in ["failed", "cancelled", "expired"]:
                        raise Exception(f"Scoring run failed: {run_status.status}")
                    time.sleep(1)

                messages = client.beta.threads.messages.list(thread_id=thread.id)
                response_text = messages.data[0].content[0].text.value.strip()

                try:
                    score = float(response_text.split("Employability Score:")[-1].split()[0])
                    reason = response_text.split("Reason:")[-1].strip()
                except Exception:
                    score, reason = 0.0, "Parsing failed"
                scores.append(score)
                reasons.append(reason)
                print(f"Score: {score}, Reason: {reason[:50]}...")

                # Only process jobs with score > 6.5 for cover letter and CV edits
                if score > 6.5:
                    cover_letter_prompt = f"""
                    Role: You are an experienced recruiter who writes persuasive, tailored, short cover letters.
                    Task: Use the user's CV pdf (via file_search) and the following job description to write a compelling cover letter (1 page maximum,
                    only ue the most important parts of the cv and the job description).

                    === Job Description ===
                    {job_desc}

                    Output Format:
                    Cover Letter: [Start your letter here]
                    """
                    client.beta.threads.messages.create(
                        thread_id=thread.id, role="user", content=cover_letter_prompt
                    )
                    run = client.beta.threads.runs.create(
                        thread_id=thread.id,
                        assistant_id=assistant_id
                    )

                    start_time = time.time()
                    while True:
                        if time.time() - start_time > max_wait_time:
                            raise Exception("Cover letter run timed out after 2 minutes")

                        run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
                        if run_status.status == "completed":
                            break
                        elif run_status.status in ["failed", "cancelled", "expired"]:
                            raise Exception(f"Cover letter run failed: {run_status.status}")
                        time.sleep(1)

                    messages = client.beta.threads.messages.list(thread_id=thread.id)
                    cover_letter_text = messages.data[0].content[0].text.value.strip()

                    # --- Suggest CV Edits ---
                    cv_edit_prompt = f"""
                    Role: You are an expert CV editor. Use the user's CV pdf (via file_search) and the job description
                    below to suggest edits that align it better.

                    === Job Description ===
                    {job_desc}

                    Output Format:
                    Proposed Edits: [Only the edits you'd suggest]
                    """
                    client.beta.threads.messages.create(
                        thread_id=thread.id, role="user", content=cv_edit_prompt
                    )
                    run = client.beta.threads.runs.create(
                        thread_id=thread.id,
                        assistant_id=assistant_id
                    )

                    start_time = time.time()
                    while True:
                        if time.time() - start_time > max_wait_time:
                            raise Exception("CV edit run timed out after 2 minutes")

                        run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
                        if run_status.status == "completed":
                            break
                        elif run_status.status in ["failed", "cancelled", "expired"]:
                            raise Exception(f"CV edit run failed: {run_status.status}")
                        time.sleep(1)

                    messages = client.beta.threads.messages.list(thread_id=thread.id)
                    edits_text = messages.data[0].content[0].text.value.strip()

                    # Store results for high-scoring jobs
                    cover_letters.append(cover_letter_text)
                    cv_edits.append(edits_text)
                else:
                    # Add placeholders for low-scoring jobs
                    cover_letters.append("")
                    cv_edits.append("")

            except Exception as e:
                print(f"Error processing job {idx+1}: {str(e)}")
                if len(scores) <= idx:
                    scores.append(0.0)
                    reasons.append(f"Error: {str(e)}")
                if len(cover_letters) <= idx:
                    cover_letters.append("")
                    cv_edits.append("")
            finally:
                try:
                    client.beta.threads.delete(thread_id=thread.id)
                    print(f"Successfully deleted thread {thread.id}")
                except Exception as thread_error:
                    print(f"Warning: Failed to delete thread {thread.id}: {str(thread_error)}")

        job_data_df["employability_score"] = scores
        job_data_df["score_reason"] = reasons
        job_data_df["cover_letter"] = cover_letters
        job_data_df["proposed_edits"] = cv_edits

        job_data_df_filtered = job_data_df[job_data_df["employability_score"] > 6.5].copy()

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        all_jobs_filename = f"all_scored_jobs_{timestamp}.csv"
        filtered_filename = f"high_scoring_jobs_{timestamp}.csv"

        job_data_df.to_csv(all_jobs_filename, index=False)
        job_data_df_filtered.to_csv(filtered_filename, index=False)

        return {
            "status": "success",
            "all_jobs_filename": all_jobs_filename,
            "filtered_filename": filtered_filename,
            "total_jobs": len(job_data_df),
            "high_scoring_jobs": len(job_data_df_filtered),
            "top_3_matches": job_data_df.sort_values("employability_score", ascending=False).head(3)[["employability_score", "score_reason"]].to_dict(orient="records")
        }

    except Exception as e:
        error_details = traceback.format_exc()
        print(f"Critical error: {str(e)}\n{error_details}")
        return {"status": "error", "message": str(e), "details": error_details}

In [6]:
def create_customized_cvs(job_data_file: str, assistant_id: str) -> dict:

    try:
        output_dir = Path("downloads")
        output_dir.mkdir(exist_ok=True)

        job_df = pd.read_csv(job_data_file)
        print(f"Loaded {len(job_df)} jobs from {job_data_file}")

        if len(job_df) == 0:
            return {"status": "warning", "message": "No jobs found in the file"}

        if "proposed_edits" not in job_df.columns:
            return {"status": "error", "message": "The jobs file doesn't contain 'proposed_edits' column"}

        created_cvs = []
        errors = []

        for index, row in job_df.iterrows():
            company_name = row.get("company", f"Company_{index}")
            job_title = row.get("title", "")
            proposed_edits = row["proposed_edits"]

            if not proposed_edits or pd.isna(proposed_edits) or proposed_edits.strip() == "":
                print(f"Skipping job {index+1} - No proposed edits available")
                errors.append(f"Job {index+1} ({company_name}): No proposed edits available")
                continue

            print(f"Processing job {index+1}/{len(job_df)} - {company_name}")

            safe_company_name = ''.join(c if c.isalnum() else '_' for c in company_name)
            cv_filename = f"CV_Ishaan_Aryan_{safe_company_name}"

            thread = client.beta.threads.create()

            try:
                prompt_user = (
                    f"Task: Create a customized CV for a job application.\n\n"
                    f"Company: {company_name}\n"
                    f"Job Title: {job_title}\n\n"
                    f"Use these proposed edits to modify the CV using code_interpreter:\n\n{proposed_edits}\n\n"
                    f"Instructions:\n"
                    f"1. Load the user's CV docx from the file store (use file_search to find it)\n"
                    f"2. Apply the proposed edits while maintaining the original template/formatting\n"
                    f"3. Only add from the proposed edits that you feel the user has done before.\n"
                    f"4. Be careful to preserve all formatting, header structure, and layout of the original CV\n"
                    f"5. The goal is to tailor the CV content to this specific job opportunity\n\n"
                    f"6. Save the new CV as a .docx file named '{cv_filename}'\n"
                )

                client.beta.threads.messages.create(
                    thread_id=thread.id,
                    role="user",
                    content=prompt_user
                )

                run = client.beta.threads.runs.create(
                    thread_id=thread.id,
                    assistant_id=assistant_id,
                    tool_choice={"type": "code_interpreter"}
                )

                start_time = time.time()
                max_wait_time = 180  # 3 minutes timeout

                while True:
                    if time.time() - start_time > max_wait_time:
                        raise Exception(f"Run timed out after {max_wait_time} seconds")

                    run_status = client.beta.threads.runs.retrieve(
                        thread_id=thread.id,
                        run_id=run.id
                    )

                    if run_status.status == "completed":
                        break
                    elif run_status.status in ["failed", "cancelled", "expired"]:
                        raise Exception(f"Run failed with status: {run_status.status}")

                    time.sleep(2)

                messages = client.beta.threads.messages.list(thread_id=thread.id)

                file_found = False

                for message in messages.data:
                    if message.role == "assistant" and hasattr(message, 'attachments') and message.attachments:
                        for attachment in message.attachments:
                            file_id = attachment.file_id
                            try:
                                file_info = client.files.retrieve(file_id)
                                original_filename = file_info.filename
                                print(original_filename)

                                if not original_filename.lower().endswith('.docx'):
                                    continue

                                if not cv_filename.lower().endswith('.docx'):
                                    save_filename = f"{cv_filename}.docx"
                                else:
                                    save_filename = cv_filename

                                file_stream = client.files.content(file_id)

                                download_path = output_dir / save_filename
                                with open(download_path, "wb") as f:
                                    f.write(file_stream.read())

                                print(f"✅ Successfully downloaded: {download_path}")
                                created_cvs.append({
                                    "company": company_name,
                                    "job_title": job_title,
                                    "filename": str(download_path)
                                })
                                file_found = True

                            except Exception as e:
                                error_msg = f"Error downloading file {file_id}: {str(e)}"
                                print(f"⚠️ {error_msg}")
                                errors.append(f"Job {index+1} ({company_name}): {error_msg}")

                if not file_found:
                    error_msg = "No CV file was generated by the assistant"
                    print(f"⚠️ {error_msg}")
                    errors.append(f"Job {index+1} ({company_name}): {error_msg}")

            except Exception as e:
                error_details = str(e)
                print(f"❌ Error processing job {index+1} ({company_name}): {error_details}")
                errors.append(f"Job {index+1} ({company_name}): {error_details}")

            finally:
                try:
                    client.beta.threads.delete(thread_id=thread.id)
                except Exception as thread_error:
                    print(f"Warning: Failed to delete thread {thread.id}: {str(thread_error)}")

        return {
            "status": "success",
            "total_jobs": len(job_df),
            "cvs_created": len(created_cvs),
            "created_files": created_cvs,
            "errors": errors if errors else None,
            "output_directory": str(output_dir)
        }

    except Exception as e:
        error_details = traceback.format_exc()
        print(f"Critical error: {str(e)}\n{error_details}")
        return {"status": "error", "message": str(e), "details": error_details}

In [7]:
tool_schema = [
  {
    "type": "function",
    "function": {
      "name": "scrape_linkedin_jobs",
      "description": "Scrapes LinkedIn job postings based on search criteria and saves them to a CSV file. Returns a dictionary with the filename, status, job count, search query used, and locations searched.",
      "parameters": {
        "type": "object",
        "properties": {
          "search_query": {
            "type": "string",
            "description": "Search term like 'Software Engineer' or 'Data Scientist'"
          },
          "location_list": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of locations to search for jobs (e.g. ['New York', 'San Francisco'])"
          },
          "job_limit": {
            "type": "integer",
            "default": 20,
            "description": "Maximum number of jobs to scrape"
          },
          "time_filter": {
            "type": "string",
            "enum": ["DAY", "WEEK", "MONTH", "ANY"],
            "description": "Filter jobs by posting time"
          }
        },
        "required": ["search_query", "location_list"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "score_jobs_against_cv",
      "description": "Scores job descriptions from a CSV file against the user's CV.",
      "parameters": {
        "type": "object",
        "properties": {
          "job_file_name": {"type": "string", "description": "File Name of the CSV containing job data"},
          "vector_store_id": {"type": "string", "description": "ID of the vector store containing the CV"}
        },
        "required": ["job_file_name"]
      }
    }
  },
   {
        "type": "function",
        "function": {
            "name": "create_customized_cvs",
            "description": "Create customized CV files for each high-scoring job based on proposed edits",
            "parameters": {
                "type": "object",
                "properties": {
                    "job_data_file": {
                        "type": "string",
                        "description": "Path to the CSV file containing high-scoring jobs with proposed edits"
                    },
                    "assistant_id": {
                        "type": "string",
                        "description": "ID of the OpenAI assistant with code interpreter capabilities"
                    }
                },
                "required": ["job_data_file", "assistant_id"]
            }
        }
   }
]

In [8]:

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def process_files_to_vector_store(input_dir="content/input", vector_store_name="CV_Collection"):
    try:
        API_KEY = userdata.get('API_KEY')
        client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", API_KEY))

        if not os.path.exists(input_dir):
            logger.error(f"Input directory '{input_dir}' does not exist")
            return {"status": "error", "message": f"Directory '{input_dir}' not found"}

        file_paths = glob.glob(f"{input_dir}/*")
        if not file_paths:
            logger.warning(f"No files found in '{input_dir}'")
            return {"status": "error", "message": "No files found in input directory"}

        logger.info(f"Found {len(file_paths)} files in '{input_dir}'")

        vector_store = client.vector_stores.create(name=vector_store_name)
        logger.info(f"Created vector store: {vector_store_name} (ID: {vector_store.id})")

        file_ids = {}

        for file_path in file_paths:
            file_name = os.path.basename(file_path)
            file_type = os.path.splitext(file_name)[1].lower()[1:]  # Get extension without dot

            key = f"{file_type} {os.path.splitext(file_name)[0]}"

            logger.info(f"Processing file: {file_name}")

            try:
                with open(file_path, "rb") as file_obj:
                    file = client.files.create(file=file_obj, purpose="assistants")
                logger.info(f"Uploaded file: {file_name} (ID: {file.id})")

                vector_store_file = client.vector_stores.files.create_and_poll(
                    vector_store_id=vector_store.id,
                    file_id=file.id,
                )

                progress_reported = False
                while vector_store_file.status == "in_progress":
                    if not progress_reported:
                        logger.info(f"Processing file {file_name}...")
                        progress_reported = True
                    time.sleep(2)

                    vector_store_file = client.vector_stores.files.retrieve(
                        vector_store_id=vector_store.id,
                        file_id=file.id
                    )

                if vector_store_file.status == "completed":
                    logger.info(f"Successfully added {file_name} to vector store")
                    file_ids[key] = file.id
                else:
                    logger.error(f"Failed to add {file_name} to vector store: {vector_store_file.status}")

            except Exception as e:
                logger.error(f"Error processing file {file_name}: {str(e)}")

        if not file_ids:
            logger.warning("No files were successfully processed")
            return {"status": "error", "message": "No files were successfully processed", "vector_store_id": vector_store.id}

        logger.info(f"Successfully processed {len(file_ids)} files")
        return {
            "status": "success",
            "file_ids": file_ids,
            "vector_store_id": vector_store.id,
            "file_count": len(file_ids)
        }

    except Exception as e:
        logger.exception(f"Error in process_files_to_vector_store: {str(e)}")
        return {"status": "error", "message": str(e)}


In [9]:
result = process_files_to_vector_store()
print("\nFinal result:")
print(f"Status: {result['status']}")

if result['status'] == 'success':
  print(f"Vector Store ID: {result['vector_store_id']}")
  print(f"Processed {result['file_count']} files")
  print("\nFile IDs:")
  for name, file_id in result['file_ids'].items():
    print(f"  {name}: {file_id}")

vector_store_id = result["vector_store_id"]
file_ids = []
for file_id in result['file_ids'].values():
  file_ids.append(file_id)


Final result:
Status: success
Vector Store ID: vs_6817bf3df01081919a58e5b8129f6f1a
Processed 2 files

File IDs:
  pdf CV_Aryan_Ishaan: file-LkHUhoGtEorGnCLYaGzMNB
  docx CV_Ishaan Aryan: file-2aGecAwQqJYBANkYcAog9F


In [15]:
API_KEY = userdata.get('API_KEY')
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", API_KEY))

assistant = client.beta.assistants.create(
    name="Job Application Evaluator1",
    instructions=(
        "Role: You are an experienced recruiter at a top-tier executive placement firm specializing in precision candidate-job matching. "
        "Task: Use the uploaded CV to evaluate job descriptions for best-fit potential. "
        "Evaluate any shared job posting based on: Relevance (skills, industry, seniority), Growth potential, and Competitive edge. "
        "Score the match on a 1-10 scale with a reason. Format: Employability Score: [X.X] Reason: [short reason]."
    ),
    model="gpt-4o-mini",
    tools=[
        {"type": "file_search"},
        {"type": "code_interpreter"},
        *tool_schema
    ],
    tool_resources={
        "file_search": {"vector_store_ids": [vector_store_id]},
        "code_interpreter": {"file_ids": [result['file_ids']['docx CV_Ishaan Aryan']]}
    }
)

print(f"Assistant created with ID: {assistant.id}")

Assistant created with ID: asst_kIGs8xxN5MEsuGKKGH3iPCAL


In [16]:
# This needs to be run first before interacting with the assistant

ASSISTANT_ID = assistant.id
# Create a thread
thread = client.beta.threads.create()
print(f"Created thread: {thread.id}")

# Function to print message content
def print_message_content(message_content):
    for content_block in message_content:
        if content_block.type == "text":
            print(f"📝 Text: {content_block.text.value}")
        elif content_block.type == "image_file":
            print(f"🖼️ Image: {content_block.image_file.file_id}")
        elif content_block.type == "tool_use":
            # For tool use, show more details because this is where most debugging info will be
            print(f"🔧 Tool Use: {content_block.tool_use.name}")
            print(f"🔧 Tool Input: {content_block.tool_use.input}")
            print(f"🔧 Tool ID: {content_block.tool_use.id}")

# Function to handle assistant interactions with debugging
def interact_with_assistant(prompt, assistant_id=ASSISTANT_ID, thread_id=thread.id):
    print(f"\n🔵 User: {prompt}")

    # Add the message to the thread
    message = client.beta.threads.messages.create(
        thread_id=thread_id,
        role="user",
        content=prompt
    )

    # Run the assistant on the thread
    run = client.beta.threads.runs.create(
        thread_id=thread_id,
        assistant_id=assistant_id
    )

    print(f"🔄 Run started with ID: {run.id}")
    name_of_job_file = ""
    # Poll for updates and print status
    while True:
        run = client.beta.threads.runs.retrieve(
            thread_id=thread_id,
            run_id=run.id
        )
        print(f"🔄 Run status: {run.status}")

        # Check for tool calls that are in progress to show debugging info
        if run.status == "requires_action":
            tool_calls = run.required_action.submit_tool_outputs.tool_calls

            # Process each tool call
            tool_outputs = []

            for tool_call in tool_calls:
                print(f"\n🔧 Tool call: {tool_call.function.name}")
                print(f"🔧 Tool input: {tool_call.function.arguments}")

                # Execute the function (in a real implementation, you'd call the actual function)
                # For debugging purposes, we'll simulate the execution
                if tool_call.function.name == "scrape_linkedin_jobs":
                    args = json.loads(tool_call.function.arguments)
                    print(f"\n⚙️ Executing {tool_call.function.name} with arguments:")
                    print(f"📋 Search query: {args.get('search_query')}")
                    print(f"📍 Locations: {args.get('location_list')}")
                    print(f"🔢 Job limit: {args.get('job_limit', 20)}")
                    result = scrape_linkedin_jobs(args.get('search_query'),args.get('location_list'),args.get('job_limit', 20))
                    print(f"📊 Function result: {result}")
                    name_of_job_file = result.get('filename')
                    print(name_of_job_file)
                    tool_outputs.append({
                        "tool_call_id": tool_call.id,
                        "output": json.dumps(result)
                    })
                elif tool_call.function.name == "score_jobs_against_cv":
                    args = json.loads(tool_call.function.arguments)
                    print(f"\n⚙️ Executing {tool_call.function.name} with arguments:")
                    print(f"📁 Job File Name: {name_of_job_file}")
                    result = score_jobs_against_cv(
                        job_file_name=name_of_job_file,
                        assistant_id=assistant_id
                    )
                    print(f"📊 Function result: {result}")
                    name_of_job_file = result.get('filtered_filename')

                    tool_outputs.append({
                        "tool_call_id": tool_call.id,
                        "output": json.dumps(result)
                    })

                elif tool_call.function.name == "create_customized_cvs":
                    args = json.loads(tool_call.function.arguments)
                    print(f"\n⚙️ Executing {tool_call.function.name} with arguments:")
                    print(f"📁 Job File Name: {name_of_job_file}")
                    result = create_customized_cvs(
                        job_data_file=name_of_job_file,
                        assistant_id=assistant_id
                    )
                    print(f"📊 Function result: {result}")

                    tool_outputs.append({
                        "tool_call_id": tool_call.id,
                        "output": json.dumps(result)
                    })

                else:
                    raise ValueError(f"Unknown function: {tool_call.function.name}")

            # Submit the outputs back to the assistant
            run = client.beta.threads.runs.submit_tool_outputs(
                thread_id=thread_id,
                run_id=run.id,
                tool_outputs=tool_outputs
            )

        # If run is completed, get messages
        if run.status == "completed":
            messages = client.beta.threads.messages.list(
                thread_id=thread_id
            )

            # Get the latest assistant message
            assistant_messages = [msg for msg in messages.data if msg.role == "assistant"]
            if assistant_messages:
                latest_msg = assistant_messages[0]
                print(f"\n🤖 Assistant ({latest_msg.created_at}):")
                print_message_content(latest_msg.content)
            break

        # Handle failed runs
        elif run.status == "failed":
            print(f"❌ Run failed: {run.last_error}")
            break

        # Handle cancelled runs
        elif run.status == "cancelled":
            print("❌ Run was cancelled")
            break

        # Wait before polling again
        time.sleep(1)

    return run

Created thread: thread_QcdFQObJ2vejL2RSnTEguKPS


In [17]:
run = interact_with_assistant("Get me 10 Software Engineering jobs in Ireland and score them based on my CV. Also update my \
 CV for the best matched jobs.")


🔵 User: Get me 10 Software Engineering jobs in Ireland and score them based on my CV. Also update my CV for the best matched jobs.
🔄 Run started with ID: run_2ATsuzDwEu8zDWSwcQ6niHVy
🔄 Run status: queued
🔄 Run status: in_progress


INFO:li:scraper:('Using strategy AnonymousStrategy',)
INFO:li:scraper:('Starting new query', "Query(query=Software Engineer options=QueryOptions(limit=10 locations=['Ireland'] filters=QueryFilters(relevance=RelevanceFilters.RECENT time=TimeFilters.DAY type=[<TypeFilters.FULL_TIME: 'F'>, <TypeFilters.INTERNSHIP: 'I'>] experience=[<ExperienceLevelFilters.ENTRY_LEVEL: '2'>, <ExperienceLevelFilters.ASSOCIATE: '3'>]) apply_link=False skip_promoted_jobs=False page_offset=0))")


🔄 Run status: requires_action

🔧 Tool call: scrape_linkedin_jobs
🔧 Tool input: {"search_query":"Software Engineer","location_list":["Ireland"],"job_limit":10}

⚙️ Executing scrape_linkedin_jobs with arguments:
📋 Search query: Software Engineer
📍 Locations: ['Ireland']
🔢 Job limit: 10


INFO:li:scraper:('Chrome debugger url', 'http://localhost:45103')
INFO:li:scraper:('Websocket debugger url: ', 'ws://localhost:45103/devtools/page/8A226F40F8E4A22C5E28193DD432320D')
INFO:li:scraper:('[Software Engineer][Ireland]', 'Opening https://www.linkedin.com/jobs/search?keywords=Software+Engineer&location=Ireland&sortBy=DD&f_TPR=r86400&f_JT=F%2CI&f_E=2%2C3&start=0')
INFO:li:scraper:('[Software Engineer][Ireland]', 'Trying first selectors set')
INFO:li:scraper:('[Software Engineer][Ireland]', 'Trying second selectors set')
INFO:li:scraper:('[Software Engineer][Ireland]', 'OK')
INFO:li:scraper:('[Software Engineer][Ireland]', 'Starting pagination loop')
INFO:li:scraper:('[Software Engineer][Ireland]', 'Found 5 jobs')
INFO:li:scraper:('[Software Engineer][Ireland][1]', 'Processed')
INFO:li:scraper:('[Software Engineer][Ireland][2]', 'Processed')
INFO:li:scraper:('[Software Engineer][Ireland][3]', 'Processed')
INFO:li:scraper:('[Software Engineer][Ireland][4]', 'Processed')
INFO:li:s

📊 Function result: {'filename': 'linkedin_jobs_Software_Engineer_20250504_193555.csv', 'status': 'success', 'job_count': 5, 'search_query': 'Software Engineer', 'locations': ['Ireland']}
linkedin_jobs_Software_Engineer_20250504_193555.csv
🔄 Run status: in_progress
🔄 Run status: in_progress
🔄 Run status: requires_action

🔧 Tool call: score_jobs_against_cv
🔧 Tool input: {"job_file_name":"linkedin_jobs_Software_Engineer_20250504_193555.csv"}

⚙️ Executing score_jobs_against_cv with arguments:
📁 Job File Name: linkedin_jobs_Software_Engineer_20250504_193555.csv
Processing 5 jobs from linkedin_jobs_Software_Engineer_20250504_193555.csv
Processing job 1/5
Score: 8.7, Reason: Strong expertise in Kubernetes, cloud environments...
Successfully deleted thread thread_YuiAvNAGc3FwDdAiUivi8nvS
Processing job 2/5
Score: 9.2, Reason: Strong alignment with required skills in Kubernete...
Error processing job 2: CV edit run timed out after 2 minutes
Successfully deleted thread thread_6vkYQToT8L6aB1aQa3